# Triaxial compression — single strain level

Analysis of **one** applied-strain level of a `triaxial_compression` run (a single-level run, or one
level picked out of a sweep). Set `LEVEL` and the run identifiers in **Config**, run the **sync** cell
once, then run everything. All analysis code lives in `scripts/lib/triaxial.py`; this notebook only
configures it and draws the eleven figures. Method notes, file list and the physics behind each
estimate are collected in **Notes** at the very end.


## 1 · Setup and computation

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

LIB = Path('lib').resolve()            # scripts/lib: triaxial.py (all analysis code) + volfrac.py
if str(LIB) not in sys.path:
    sys.path.insert(0, str(LIB))
import triaxial as tri
tri = importlib.reload(tri)            # pick up edits to lib/triaxial.py without a kernel restart
tri.setup_style()
print('analysis code: ', LIB / 'triaxial.py')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit when switching runs
# ══════════════════════════════════════════════════════════════════════════
LEVEL = "0.10"      # the ONE applied-strain level analysed here, as a string.  It must be one of the
                    # STRAIN_TARGETS in triaxial_compression.batch (files are tagged _c<LEVEL>).
                    # For a sweep run just pick the level you want, e.g. "0.15".
cfg = tri.Config(
    DATANAME    = "final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000000",
    INTERACTION = "1.0_1.0",          # epsSS_epsSP
    NSTEPS      = 5000000,            # production steps; None -> auto-detect the largest present locally
    RUN_ID      = "periodic_rho04_1.4M_5M_twoPlatesMove_3",   # local folder under flow_data_local/{compression,plots}
    COMP_LEVELS = [LEVEL],
    # ---- measurement windows (both halved 2026-09-02, see Notes) ----
    plateau_frac      = 0.25,   # FIXED trailing fraction of the hold for the profile plateau means
    plateau_frac_auto = 0.45,   # LONGEST candidate trailing window for the piston plateau (auto drift test)
    # ---- solvent volume fractions (lib/volfrac.py) ----
    VOR_ENABLE = True, REF_VOR_FRAMES = 3, VOR_MAX_FRAMES = 4, P_CAL = 1.5,
    # ---- G from the lateral network stress ----
    G_SUBTRACT_REF = True,      # use sigma' increments relative to the eps = 0 reference state
    # ---- D_c consolidation fit + hold-adequacy check ----
    DC_FREE_AMPS = True, DC_N_MODES = 5, DC_TRIM_BINS = 2, DC_SLOW_REF = 0.17, DC_TARGET_RESID = 0.01,
)
# Every other knob keeps its default (binWidth, n_curves, Ncount_min, dt_lj, ci_level, gel_thresh,
# flat_tol, wall_margin, baseline_zf, roll_win, VOR_NORM, PHI_FLOOR, DC_*, Expanse host/paths):
# see `tri.Config` in lib/triaxial.py, or pass them here as extra keyword arguments.

In [ ]:
# Pull the files this notebook reads from Expanse in ONE login (password + TOTP prompts).
# Logs in when ANY data file is missing locally (files a previous sync found absent on the
# cluster are remembered and do not re-prompt); FORCE_SYNC=True re-checks everything and
# also refreshes the trajectories.  SYNC=False never touches the network.
SYNC, FORCE_SYNC = True, False
if SYNC:
    tri.sync_from_expanse(cfg, levels=[LEVEL], force=FORCE_SYNC)

In [ ]:
# Load + compute everything (a few seconds, plus ~20 s per Voronoi frame):
#   R  -- the eps = 0 reference state: geometry, reference profiles, gel bounds
#   L  -- this level: stresses, Terzaghi split, piston plateau, M, G, D_c, kappa
R = tri.load_reference(cfg)
L = tri.load_level(cfg, R, LEVEL)
assert L is not None, f'level {LEVEL}: core files missing -- run the sync cell'
tri.add_volume_fractions(cfg, R, [L])       # mass-fraction / Voronoi / lambda-calibrated phi_s
tri.print_summary(cfg, [L])

## 2 · Figures

In [ ]:
# 1 · Strain diagnostic -- solid ε_Rg, dashed ε_BB, faint ε_piston (diagnostic only); shaded = plateau window
tri.fig_strain(cfg, R, [L]);

In [ ]:
# 2 · Solvent volume fraction φ_s: mass fraction, Voronoi and λ-calibrated Voronoi -- reference (ε = 0) vs compressed
tri.fig_volfrac(cfg, R, L);

In [ ]:
# 3 · Total stress evolution σ^t_zz, σ^t_xx, σ^t_yy -- reference (dashed) -> hold (cividis) -> plateau (bold)
tri.fig_total_stress(cfg, R, L);

In [ ]:
# 4 · Solvent and polymer partial σ_zz evolutions with the total superimposed
tri.fig_partial_stress(cfg, R, L);

In [ ]:
# 5 · Network stress evolution σ'_zz, σ'_xx, σ'_yy (Terzaghi: σ' = σ^t − p_pore), 95 % bands
tri.fig_network_stress(cfg, R, L);

In [ ]:
# 6 · Piston pressure P = F_z/A vs step, linear + log; green = auto-selected plateau window
tri.fig_piston(cfg, R, L);

In [ ]:
# 7 · Longitudinal modulus M: network (σ'_zz/ε) vs piston (P/ε), 95 % CIs
tri.fig_M(cfg, R, L);

In [ ]:
# 8 · Network-stress anisotropy σ'_zz/σ'_xx and σ'_zz/σ'_yy vs step  (= M/(M − 2G))
tri.fig_ratio(cfg, R, L);

In [ ]:
# 9 · Shear modulus G = (σ'_zz − σ'_ii)/(2ε) from xx and from yy (should agree by symmetry)
tri.fig_G(cfg, R, L);

In [ ]:
# 10 · Cooperative diffusivity D_c: consolidation fit of u_z(ζ, t)/L
tri.fig_Dc(cfg, R, L);

In [ ]:
# 11 · κ = D_c/M (hydraulic permeability / viscosity) from the network and the piston M
tri.fig_kappa(cfg, R, L);


## Notes

Everything below is reference material — nothing above depends on reading it.

### How to use this notebook

1. **Config**: `DATANAME`, `INTERACTION`, `NSTEPS` name the run exactly as LAMMPS tagged its output
   files (`<DATANAME>_<INTERACTION>_<NSTEPS>`); `RUN_ID` is the local folder under
   `flow_data_local/{compression,plots}/`. Every production file carries a `_c<level>` tag (even a
   single-level run), `*_ref` files are shared across levels. The measurement knobs that matter for
   $M$, $G$, $D_c$, $\kappa$ are spelled out in the Config cell; everything else keeps the default in
   `tri.Config` (`lib/triaxial.py`).
2. **Sync**: one Expanse login pulls every file the notebook reads. It logs in whenever a data file is
   missing locally (files the cluster does not have are remembered in `.sync_absent.json` and stop
   prompting); `FORCE_SYNC=True` re-checks everything and refreshes the trajectories as well.
3. **Load**: `load_reference` builds the $\varepsilon=0$ state `R`, `load_level` builds a level dict
   `L` (stresses, Terzaghi split, piston plateau, $M$, $G$, $D_c$, $\kappa$), `add_volume_fractions`
   adds the Voronoi / calibrated $\phi_s$ (the only slow step, ~20 s per tessellated frame),
   `print_summary` prints the headline numbers and the hold-adequacy check.
4. **Figures**: one function call each; every figure is also saved as a PNG in `PLOT_DIR`.

`lib/triaxial.py` is shared with the other triaxial-compression notebook so the definitions cannot
drift between them; `importlib.reload(tri)` in the first cell picks up edits without a kernel restart.

### Files read

Into `flow_data_local/compression/<RUN_ID>/` (cluster `output_files/…`):

| file pattern | content |
|---|---|
| `sigma{zz,xx,yy}_{polymer,solvent}[_ref]_<sim>[_c<lvl>].dat` | group partial stress profiles (`compute stress/atom NULL` → **kinetic term included**), per bin volume |
| `solvent_density_z[_ref]_…` | solvent number / mass density profiles |
| `strain_zz_…`, `strain_piston_…`, `gel_dimensions_{bb,rg}_…`, `polymer_com_…`, `gel_edges_…` | strain and geometry diagnostics (fine `strain_freq` cadence) |
| `piston_position_…`, `piston_force_…`, `piston_force_avg_…` | piston $z(t)$, raw and LAMMPS block-averaged $F_z(t)$ |
| `box_dimensions_…` | $l_x l_y$ cross-section (fixed; fallback = dump box header) |
| `disp_z_polymer_…` | polymer $u_z(z,t)$ during the hold, for $D_c$ |

Into `flow_data_local/traj_files.nosync/`: `traj_ref_<sim>.lammpstrj` (box header, wall planes,
reference Voronoi frames) and `traj_stress_<sim>_c<lvl>.lammpstrj` (plateau Voronoi frames). The
pair/bond dumps of the long-form notebook are **not** needed here.

### Coordinates, gel bounds, windows

* $z$-bins: `binWidth` = 2 σ, bin centres $z = z_{lo} + (i-\tfrac12)\,$`binWidth`, plotted as $z/L_z$.
  Support (type 4, solid line) and piston (type 5, dash-dot) planes come from the first `traj_ref`
  frame; the piston's held position per level from `piston_position`.
* Gel interior (reference) = bins where $|\sigma_{p,zz}^{\rm ref}| >$ `gel_thresh` × max; membrane
  (per level) = the same test on the **final** production polymer stress, fixed for every curve of
  that level. "Interior" means trimmed by `wall_margin` = 4 σ off both ends.
* Plateau window = the last `plateau_frac` = 25 % of the production hold (`halt_ts`); evolution plots
  show the whole hold (`n_curves` snapshots, cividis, final curve bold black).

### Solvent volume fraction $\phi_s$ (figure 2)

* **mass fraction** $\phi_s^{\rm mf} = \rho_s(z)/\rho_{s,0}$, with $\rho_{s,0}$ the bulk-reservoir
  density of the **reference** state (incompressible-solvent assumption).
* **Voronoi** $\phi_s^{\rm vor}$: periodic voro++ tessellation of the mobile beads (types 1, 2, 3),
  solvent cell volume per bin over $V_{\rm bin}$ (`VOR_NORM='bin'`), from `lib/volfrac.py`.
* **λ-calibrated** $\phi_s^{\rm cal} = \min(1,\ \lambda(\varphi_p^{\rm vor}, P_{\rm CAL})\,
  \phi_s^{\rm vor,mobile})$ — the packing correction fit on the calibration sweep
  (`scripts/calibration/calibration_lambda.json`, `calibration_analysis.ipynb`). `P_CAL` is used
  for both states; the compressed state strictly sits in $[P_{\rm CAL}, P_{\rm CAL}+P_{\rm piston}]$,
  so rerun with the other bracket end for a $P$-sensitivity bar.
* Reference: `REF_VOR_FRAMES` frames spread over the reference window (mean ± 95 % CI across frames).
  Compressed: `VOR_MAX_FRAMES` frames inside the plateau window (plateau mean ± CI); the mass-fraction
  curve uses every density snapshot in the plateau window.

### Stresses (figures 3–5)

* Group profiles from `compute stress/atom NULL` **include** the kinetic term $n(z)k_BT$ (NULL only
  suppresses the velocity-bias temperature compute). Total $\sigma^t = \sigma_p + \sigma_s$.
* **Terzaghi split** $\sigma^t_{ii} = \sigma'_{ii} + p_{\rm pore}$: the pore pressure is read **per
  curve and per component** from the flat far-reservoir window $z/L_z \in$ `baseline_zf` ± `baseline_zf_half`
  (0.91–0.99, extreme-edge bin dropped); the network stress is the jump of the total above it. In the
  reservoir the fluid is isotropic, so the three baselines agree to noise (printed by `load_reference`).
* 95 % bands on $\sigma'$: per-bin single-snapshot noise (scatter of the reference snapshots) ⊕ the
  baseline uncertainty, in quadrature.
* The reference $\sigma'$ profile of each component is drawn dashed as the starting point of every
  evolution; $\sigma'_{zz}$ must be ≈ 0 at $\varepsilon=0$ (mechanical equilibrium with the bath),
  but the **lateral** $\sigma'_{xx},\sigma'_{yy}$ need not be — the periodic box fixes $l_x, l_y$.

### Piston pressure and the plateau window (figure 6)

$P(t) = F_z(t)/A$ with $A = l_x l_y$ fixed (no lateral barostat); the piston is undamped, so the
force is thermally noisy **and autocorrelated**. The quoted plateau $\langle P\rangle$ is a **circular
block bootstrap** (block ≈ 2 τ_int, 2000 resamples, 95 % percentile CI) over the **longest
drift-free trailing window**: candidates `plateau_frac_auto` × 9/9, 8/9, … 2/9 of the series (the
last 45 %, 40 %, … 10 % at the default 0.45); a window passes when its two halves agree within their
block-bootstrap CIs; the shortest is used, with a warning, if none passes. The LAMMPS block-averaged
series (`piston_force_avg`) is preferred over the raw print samples because it resolves the slow force
autocorrelation. Both `plateau_frac` (0.50 → 0.25) and `plateau_frac_auto` (0.90 → 0.45) were halved
on 2026-09-02 because a still-consolidating hold had pulled the decaying transient into the mean and
biased $M_{\rm piston}$ ~6 % high; the real fix is a hold that outlasts the consolidation time (see
the hold check below).

### Longitudinal modulus $M$ (figure 7)

Strain-controlled: the piston is driven to a displacement $\varepsilon\,L_0^{BB}$, so the **applied**
strain (the `_c<level>` value) is the denominator. The measured $\varepsilon_{Rg}$ / $\varepsilon_{BB}$
(figure 1) are consistency checks only: $\varepsilon_{BB}$ should ≈ the applied value, confirming the
plates bracket the gel; $\varepsilon_{\rm piston}$ over-reads (its $L_0$ includes the solvent void).

* $M_{\rm network} = \langle\sigma'_{zz}\rangle_{\rm membrane}/\varepsilon$ from the final
  (plateau-averaged) network profile; CI = 95 % t-interval over the membrane bins.
* $M_{\rm piston} = \langle P\rangle_{\rm plateau}/\varepsilon$; CI propagated from the block bootstrap.

Agreement of the two validates the Terzaghi split.

### Shear modulus $G$ from the network-stress anisotropy (figures 8–9)

Uniaxial strain with the lateral box fixed ($\varepsilon_{xx}=\varepsilon_{yy}=0$) gives, for an
isotropic drained network with Lamé constants $\lambda, G$ and $M = \lambda + 2G$:

$$\sigma'_{zz} = M\,\varepsilon,\qquad \sigma'_{xx} = \sigma'_{yy} = \lambda\,\varepsilon = (M-2G)\,\varepsilon
\quad\Longrightarrow\quad \frac{\sigma'_{zz}}{\sigma'_{xx}} = \frac{M}{M-2G},\qquad
G = \frac{\sigma'_{zz}-\sigma'_{xx}}{2\,\varepsilon}.$$

(Equivalently $\lambda = K - \tfrac23 G$ with $M = K + \tfrac43 G$; the ratio is $M/(M-2G)$.) The notebook forms $G$ per bin of the **wall-trimmed membrane interior** (the lateral total
stress jumps at the gel boundary, so the edge bins carry no anisotropy information) from the final network profiles and
quotes the bin mean with its 95 % t-interval, once from $xx$ and once from $yy$ — the two should agree
by symmetry, and their spread is a second, independent error estimate. With `G_SUBTRACT_REF=True`
(default) the network stresses are **increments** relative to the $\varepsilon=0$ reference, because the
lateral network stress of the periodic slab need not vanish before loading; the reference values are
printed. The crosses in figure 9 use $M_{\rm piston}$ instead of $\langle\sigma'_{zz}\rangle$ in
$G = (M-\lambda)/2$. Figure 8 shows the membrane-mean ratio for every snapshot of the hold, so the
approach to the plateau is visible.

### Cooperative diffusivity $D_c$ (figure 10)

Two-sided consolidation of the held gel (both plates are solvent-transparent). `triaxial_compression.lmp`
resets the displacement reference at each hold onset, so `disp_z_polymer` stores
$u_{\rm dat} = u_z(t) - u_z(t_{\rm hold})$. With $\zeta = (z - z_{\rm perm})/L$, $\tau = D_c (t-t_{\rm hold})/L^2$
and the boundary conditions $u_z(0,t)=0$ (support), $u_z(1,t) = -\Delta L/L$ (held piston):

$$\frac{u_z}{L} = -\frac{\Delta L}{L}\,\zeta + \sum_{k\ \rm odd} B_k\,[2\zeta - 1 + \cos k\pi\zeta]\,e^{-k^2\pi^2\tau},
\qquad \frac{u_{\rm dat}}{L} = T(\tau) - T(0),$$

fitted by least squares in $D_c$ (`DC_BOUNDS`) with `DC_N_MODES` free amplitudes (`DC_FREE_AMPS=True`:
the hold-onset state is fitted — the fast drive drains only thin face skins) or the uniform-$p_0$
undrained start (`False`). $L$ is the **compressed** BB thickness (the draining layer), placed at equal
contact gaps off the support and the held piston; only bins with ≥ `Ncount_min` polymer atoms, trimmed
by `DC_TRIM_BINS`, enter the fit.

**Hold-adequacy check** (printed by `print_summary`): the slowest mode decays with
$\tau_1 = L^2/(\pi^2 D_c)$; a hold of length $T$ leaves a mean residual
$r = \frac{\tau_1}{fT}[e^{-(1-f)T/\tau_1} - e^{-T/\tau_1}]$ over the last $f =$ `plateau_frac`, i.e. the
fraction by which the piston pressure still sits above its relaxed value. It is reported for the fitted
$D_c$ (dominated by the fast face-skin relaxation → an upper bound) and for `DC_SLOW_REF` = 0.17 σ²/τ,
the slow collective mode from the free-swelling equilibration that the `.lmp` sizes its holds on.

### $\kappa = D_c/M$ (figure 11)

In linear poroelasticity the consolidation coefficient is $D_c = \kappa M$ with $\kappa = k/\eta$ the
Darcy permeability over the solvent viscosity, so $\kappa = D_c/M$ (LJ units σ⁵/(ε τ)). Both $M$
estimates are used; the CI is propagated from $M$ (the fit gives $D_c$ no formal error bar).

### Colours and files

Wong (2011) categorical palette; cividis for the time gradient; the gel/membrane is shaded grey.
Figures are saved to `flow_data_local/plots/compression/<RUN_ID>/` with the stems
`strain_diagnostic` (tagged `_c<level>`), `volfrac_profiles`, `total_stress_evolution`, `partial_stress_evolution`,
`network_stress_evolution`, `piston_pressure_history`, `M_comparison`, `network_stress_ratio`,
`G_estimate`, `Dc_consolidation_fit`, `kappa` (sweep notebook: `sweep_*`, including `sweep_strain_diagnostic`). None of these names collide with the long-form notebook's files in the same folder.

*Derived 2026-09-02 from the long-form `triaxial_compression.ipynb` (which keeps the solvent-phase
stress, cross-virial and Widom-insertion diagnostics); the M / plateau / D_c estimators are unchanged.*
